# FinChart-R2 — Phase 2A: Teacher-Assisted Structured Annotation

## Objective

Phase 1 (`FinChart-R1`) established a frozen baseline and identified the main weaknesses of the compact Qwen3-VL-4B model.

Phase 2A now uses a stronger multimodal teacher **offline** to annotate ChartQA training examples with structured chart reasoning supervision.

The teacher is **not treated as ground truth**.

Every teacher annotation is checked against:

- the original ChartQA answer
- a closed task taxonomy
- a closed operation taxonomy
- deterministic arithmetic validation where possible
- confidence thresholds
- conflict rules
- manual audit

The final goal is to create a clean, auditable SFT-ready dataset for Phase 2B.

### Research flow

```text
FinChart-R1 / Phase 1
Baseline + error analysis
        ↓
Measured weaknesses
        ↓
ChartQA train
        ↓
Teacher-assisted structured annotation
        ↓
Deterministic validation + conflict filtering
        ↓
Audited SFT-ready dataset
        ↓
FinChart-R2 / Phase 2B
Qwen3-VL-4B + Unsloth QLoRA
        ↓
Frozen Phase 1 evaluation
```

### Key principle

> The teacher generates candidate supervision. The validation layer decides whether that supervision is safe enough to train on.

## Phase 2A Definition of Done

Phase 2A is complete only when:

1. Phase 1 validation boundary remains frozen.
2. Only ChartQA `train` is used for SFT data construction.
3. Teacher annotations use a closed schema and closed taxonomy.
4. Teacher output is validated instead of blindly trusted.
5. Representation-equivalent answers are handled separately from semantic conflicts.
6. Arithmetic supervision is recomputed where supported.
7. Suspicious, conflicting, malformed, or low-confidence samples are excluded or reviewed.
8. Full-dataset statistics and an audit sample are exported.
9. A final SFT-ready dataset is exported.
10. All final Phase 2A gates pass.

No QLoRA training should start before these conditions are satisfied.

## 1. Install dependencies

In [ ]:
# Uncomment in a fresh Colab runtime.
# !pip install -q -U datasets pandas requests pillow tqdm

import os
import re
import io
import json
import time
import math
import base64
import random
import statistics
from pathlib import Path

import pandas as pd
import requests

SEED = 42
random.seed(SEED)

## 2. Runtime check

In [ ]:
import platform

print("Python:", platform.python_version())
print("Phase 2A is primarily a data-generation and validation pipeline.")
print("GPU is not required unless the teacher itself runs locally.")

## 3. Mount Drive and define workspaces

- `FinChart-R1` = frozen Phase 1 baseline/evaluation artifacts
- `FinChart-R2` = Phase 2A annotation + Phase 2B SFT

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive")

PHASE1_PROJECT_DIR = DRIVE_ROOT / "FinChart-R1"
PROJECT_DIR = DRIVE_ROOT / "FinChart-R2"

# Keep every generated artifact under results/ for inspection and reproducible pilot handoff.
RESULTS_DIR = PROJECT_DIR / "results"
DATA_DIR = RESULTS_DIR
AUDIT_DIR = RESULTS_DIR / "audit"
LOG_DIR = RESULTS_DIR / "logs"
CHECKPOINT_DIR = RESULTS_DIR / "checkpoints"

for p in [DATA_DIR, RESULTS_DIR, AUDIT_DIR, LOG_DIR, CHECKPOINT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Phase 1:", PHASE1_PROJECT_DIR)
print("Phase 2:", PROJECT_DIR)

## 4. Freeze experiment boundaries

In [ ]:
BASE_MODEL = "unsloth/Qwen3-VL-4B-Instruct-unsloth-bnb-4bit"
DATASET_NAME = "HuggingFaceM4/ChartQA"

TRAINING_SPLIT = "train"

FROZEN_EVALUATION_SPLIT = "val"
FROZEN_EVALUATION_OFFSET = 0
FROZEN_EVALUATION_SAMPLES = 500

PHASE1_FINAL_CSV = (
    PHASE1_PROJECT_DIR
    / "results"
    / "qwen3vl4b_chartqa_val_0_500_phase1_final.csv"
)

assert TRAINING_SPLIT == "train"

print("SFT source:", f"{DATASET_NAME}/{TRAINING_SPLIT}")
print(
    "Frozen evaluation:",
    f"{FROZEN_EVALUATION_SPLIT}[{FROZEN_EVALUATION_OFFSET}:"
    f"{FROZEN_EVALUATION_OFFSET + FROZEN_EVALUATION_SAMPLES}]"
)

## 5. Verify Phase 1 baseline

In [ ]:
if not PHASE1_FINAL_CSV.exists():
    raise FileNotFoundError(
        f"Missing Phase 1 artifact: {PHASE1_FINAL_CSV}"
    )

phase1 = pd.read_csv(PHASE1_FINAL_CSV)

assert len(phase1) == 500

det_correct = int(
    phase1["deterministic_correct"]
    .fillna(False)
    .astype(bool)
    .sum()
)
final_correct = int((phase1["final_verdict"] == "CORRECT").sum())
final_incorrect = int((phase1["final_verdict"] == "INCORRECT").sum())

assert det_correct == 317
assert final_correct == 351
assert final_incorrect == 114

print("Phase 1 verified.")
print("Deterministic correct:", det_correct)
print("Final CORRECT:", final_correct)
print("Final INCORRECT:", final_incorrect)

## 6. Phase 1 failure profile

In [ ]:
confirmed_errors = phase1[
    phase1["final_verdict"] == "INCORRECT"
].copy()

error_counts = (
    confirmed_errors["judge_error_type"]
    .fillna("UNKNOWN")
    .value_counts()
    .rename_axis("error_type")
    .reset_index(name="count")
)

error_counts["share"] = error_counts["count"] / len(confirmed_errors)
display(error_counts)

## 7. Closed teacher taxonomy

The teacher must choose from a predefined taxonomy.

This prevents free-form labels such as `median_then_add_maximum` or `sum and comparison`, which are difficult to validate consistently.

In [ ]:
TASK_TYPES = {
    "visual_grounding",
    "numerical_reasoning",
    "counting",
    "logical_reasoning",
}

SUBTYPES = {
    # visual
    "direct_value",
    "temporal_lookup",
    "position",
    "series_or_legend",
    "color",
    "extrema",
    "intersection",
    "category_lookup",

    # numerical
    "sum",
    "difference",
    "average",
    "median",
    "ratio",
    "percentage",
    "percentage_change",
    "max_difference",
    "min_max_arithmetic",
    "multi_step",

    # counting
    "count_elements",
    "count_threshold",
    "count_occurrences",
    "count_intersections",

    # logical
    "boolean_comparison",
    "ranking",
    "trend",
    "conditional",
    "comparison",
}

OPERATIONS = {
    "none",
    "lookup",
    "sum",
    "difference",
    "average",
    "median",
    "ratio",
    "percentage",
    "percentage_change",
    "count",
    "comparison",
    "max_difference",
    "min_max",
    "multi_step",
}

## 8. Initial curriculum target

Phase 1 suggests the following starting target mix:

```text
Numerical reasoning   45%
Visual grounding      30%
Counting              15%
Logical reasoning     10%
```

This is a soft target applied after quality filtering.

In [ ]:
CURRICULUM_TARGETS = {
    "numerical_reasoning": 0.45,
    "visual_grounding": 0.30,
    "counting": 0.15,
    "logical_reasoning": 0.10,
}

assert abs(sum(CURRICULUM_TARGETS.values()) - 1.0) < 1e-9

## 9. Answer normalization

In [ ]:
WRAPPER_RE = re.compile(
    r"^\s*(?:answer|final answer|result)\s*:\s*",
    flags=re.IGNORECASE,
)

def unwrap_answer(value):
    if isinstance(value, (list, tuple)):
        if len(value) == 0:
            return ""
        if len(value) == 1:
            return value[0]
        return [str(x).strip() for x in value]
    return value


def normalize_text(value) -> str:
    if value is None:
        return ""

    value = unwrap_answer(value)

    if isinstance(value, list):
        return json.dumps(value, ensure_ascii=False)

    text = str(value).strip()
    text = WRAPPER_RE.sub("", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s*/\s*", "/", text)
    text = text.strip(" \t\n\r.,;:")
    return text


def parse_number(value):
    text = normalize_text(value)

    if not text:
        return None

    cleaned = (
        text.replace(",", "")
        .replace("$", "")
        .replace("€", "")
        .replace("£", "")
    )

    is_percent = cleaned.endswith("%")
    if is_percent:
        cleaned = cleaned[:-1].strip()

    try:
        number = float(cleaned)
    except ValueError:
        return None

    return {
        "value": number,
        "is_percent": is_percent,
    }

## 10. Answer relation: exact, representation-equivalent, or conflict

A dataset answer and teacher answer are not always string-identical.

Example:

```text
dataset: 0.77
teacher: 77%
```

This should not automatically become a semantic conflict.

We keep representation equivalence separate from semantic correction.

In [ ]:
def classify_answer_relation(dataset_answer, teacher_answer, tolerance=1e-6):
    a = normalize_text(dataset_answer).lower()
    b = normalize_text(teacher_answer).lower()

    if a == b:
        return "EXACT_MATCH"

    pa = parse_number(a)
    pb = parse_number(b)

    if pa is not None and pb is not None:
        va = pa["value"]
        vb = pb["value"]

        if abs(va - vb) <= tolerance:
            return "NUMERIC_MATCH"

        # 0.77 vs 77%
        if pa["is_percent"] != pb["is_percent"]:
            if pa["is_percent"]:
                percent_value = va
                proportion_value = vb
            else:
                percent_value = vb
                proportion_value = va

            if abs(percent_value / 100.0 - proportion_value) <= tolerance:
                return "PERCENT_PROPORTION_EQUIVALENT"

        # 0.77 vs 77 without explicit %
        if abs(va * 100 - vb) <= tolerance or abs(vb * 100 - va) <= tolerance:
            return "POSSIBLE_SCALE_EQUIVALENT"

    return "CONFLICT"

## 11. Arithmetic validation helpers

In [ ]:
def to_float_list(values):
    result = []

    for value in values or []:
        parsed = parse_number(value)
        if parsed is None:
            return None
        result.append(parsed["value"])

    return result


def recompute_operation(values, operation):
    xs = to_float_list(values)

    if xs is None or not xs:
        return None

    op = normalize_text(operation).lower()

    if op == "sum":
        return sum(xs)

    if op == "average":
        return sum(xs) / len(xs)

    if op == "median":
        return statistics.median(xs)

    if op == "difference" and len(xs) == 2:
        return xs[0] - xs[1]

    if op == "ratio" and len(xs) == 2 and xs[1] != 0:
        return xs[0] / xs[1]

    if op == "percentage" and len(xs) == 2 and xs[1] != 0:
        return xs[0] / xs[1] * 100

    if op == "percentage_change" and len(xs) == 2 and xs[0] != 0:
        return (xs[1] - xs[0]) / xs[0] * 100

    if op == "count":
        return float(len(xs))

    if op == "max_difference" and len(xs) >= 2:
        return max(xs) - min(xs)

    return None

## 12. Load ChartQA train

In [ ]:
from datasets import load_dataset

train_ds = load_dataset(
    DATASET_NAME,
    split=TRAINING_SPLIT,
)

print(train_ds)
print("Train samples:", len(train_ds))
print("Columns:", train_ds.column_names)

## 13. Inspect raw ChartQA schema

In [ ]:
sample = train_ds[0]

for key, value in sample.items():
    if key == "image":
        print(key, ":", type(value), getattr(value, "size", None))
    else:
        print(key, ":", value)

## 14. Build base records

The teacher becomes the primary task labeler.

Rule-based classification is no longer required as the main source of task labels.

In [ ]:
def build_base_record(example, dataset_index):
    question = example.get("query", example.get("question", ""))
    raw_answer = example.get("label", example.get("answer", ""))
    answer = unwrap_answer(raw_answer)

    return {
        "sample_id": f"chartqa_train_{dataset_index}",
        "dataset_index": dataset_index,
        "question": normalize_text(question),
        "dataset_answer": normalize_text(answer),
        "raw_answer": answer,
        "image_present": example.get("image") is not None,
    }

# 15. Teacher-Assisted Annotation

The teacher receives:

- chart image
- question
- dataset answer

It must independently inspect the chart and return structured supervision using the closed taxonomy.

## 15.1 Teacher configuration

In [ ]:
# Store TEACHER_API_KEY in Colab Secrets. Local execution may use .env or environment variables.
try:
    from dotenv import load_dotenv
    load_dotenv(PROJECT_DIR / ".env", override=False)
    # Backward-compatible local fallback; prefer .env and keep secrets out of version control.
    if not os.getenv("TEACHER_API_KEY"):
        load_dotenv(PROJECT_DIR / ".env.example", override=False)
except ImportError:
    pass

try:
    from google.colab import userdata
    _secret_key = userdata.get("TEACHER_API_KEY")
except (ImportError, KeyError):
    _secret_key = None

TEACHER_API_KEY = _secret_key or os.getenv("TEACHER_API_KEY", "")
TEACHER_BASE_URL = os.getenv("TEACHER_BASE_URL", "https://hhtechapi.com/v1")
TEACHER_MODEL = os.getenv("TEACHER_MODEL", "gpt-5.6-sol")

TEACHER_TIMEOUT = 120
TEACHER_MAX_RETRIES = 3
TEACHER_MIN_CONFIDENCE = 0.80

# Progression: 50 -> 500 -> None. This notebook is configured for the 500-sample pilot.
PHASE2A_BUILD_LIMIT = 500
PILOT_TARGET_SAMPLES = 500

SAVE_EVERY = 25

ANNOTATION_CHECKPOINT = (
    CHECKPOINT_DIR / f"phase2a_teacher_annotations_{PHASE2A_BUILD_LIMIT}.jsonl"
)

if not (TEACHER_API_KEY and TEACHER_BASE_URL and TEACHER_MODEL):
    raise RuntimeError(
        "Configure TEACHER_API_KEY in Colab Secrets (or environment variables) before annotation."
    )

print("Teacher configured: True")
print("Build limit:", PHASE2A_BUILD_LIMIT)
print("Checkpoint:", ANNOTATION_CHECKPOINT)

## 15.2 Image encoder

In [ ]:
def pil_image_to_data_url(image):
    if image is None:
        return None

    buffer = io.BytesIO()

    if getattr(image, "mode", None) not in {"RGB", "RGBA"}:
        image = image.convert("RGB")

    image.save(buffer, format="PNG")
    encoded = base64.b64encode(buffer.getvalue()).decode("utf-8")

    return f"data:image/png;base64,{encoded}"

## 15.3 Teacher prompt

In [ ]:
TEACHER_SYSTEM_PROMPT = f'''
You are a multimodal chart annotation teacher.

Your job is to inspect the chart image and produce structured supervision for a compact chart-reasoning model.

Return VALID JSON ONLY.

Allowed task_type values:
{sorted(TASK_TYPES)}

Allowed subtype values:
{sorted(SUBTYPES)}

Allowed operation values:
{sorted(OPERATIONS)}

Required JSON schema:
{{
  "task_type": "...",
  "subtype": "...",
  "target_series": null,
  "target_category": null,
  "relevant_values": [],
  "operation": "none",
  "calculation": null,
  "final_answer": "...",
  "confidence": 0.0
}}

Rules:
1. Inspect the chart independently.
2. Use only the allowed taxonomy.
3. relevant_values must contain only values needed to solve the question.
4. If no arithmetic is required, use operation="none".
5. Keep calculation concise and inspectable.
6. Do not output long free-form chain-of-thought.
7. Do not force agreement with the dataset answer.
8. If the chart is ambiguous or hard to read, reduce confidence.
9. final_answer must directly answer the question.
10. Output JSON only.
'''.strip()


def make_teacher_user_prompt(record):
    return f'''
Question:
{record["question"]}

Dataset answer:
{record["dataset_answer"]}

Inspect the chart independently and return the structured annotation.
'''.strip()

## 15.4 Teacher API caller

In [ ]:
def get_chat_completions_url(base_url):
    base_url = base_url.rstrip("/")

    if base_url.endswith("/chat/completions"):
        return base_url

    return base_url + "/chat/completions"


def extract_json_object(text):
    if not text:
        raise ValueError("Empty teacher response")

    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.I)
    text = re.sub(r"\s*```$", "", text)

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    start = text.find("{")
    end = text.rfind("}")

    if start == -1 or end == -1 or end <= start:
        raise ValueError("No JSON object found")

    return json.loads(text[start:end + 1])


def assert_teacher_available():
    """Fail fast before a long annotation run when the provider is unavailable."""
    response = requests.post(
        get_chat_completions_url(TEACHER_BASE_URL),
        headers={
            "Authorization": f"Bearer {TEACHER_API_KEY}",
            "Content-Type": "application/json",
        },
        json={
            "model": TEACHER_MODEL,
            "temperature": 0,
            "max_tokens": 1,
            "messages": [{"role": "user", "content": "Reply with OK."}],
        },
        timeout=min(30, TEACHER_TIMEOUT),
    )
    response.raise_for_status()


def call_teacher(record, image):
    if not (TEACHER_API_KEY and TEACHER_BASE_URL and TEACHER_MODEL):
        raise RuntimeError("Teacher API is not configured.")

    payload = {
        "model": TEACHER_MODEL,
        "temperature": 0,
        "messages": [
            {
                "role": "system",
                "content": TEACHER_SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": make_teacher_user_prompt(record),
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": pil_image_to_data_url(image)
                        },
                    },
                ],
            },
        ],
    }

    headers = {
        "Authorization": f"Bearer {TEACHER_API_KEY}",
        "Content-Type": "application/json",
    }

    url = get_chat_completions_url(TEACHER_BASE_URL)

    last_error = None

    for attempt in range(TEACHER_MAX_RETRIES):
        try:
            response = requests.post(
                url,
                headers=headers,
                json=payload,
                timeout=TEACHER_TIMEOUT,
            )
            response.raise_for_status()

            body = response.json()
            text = body["choices"][0]["message"]["content"]

            return extract_json_object(text)

        except Exception as exc:
            last_error = exc
            wait = min(2 ** attempt, 8)

            print(
                f"Teacher retry {attempt + 1}/"
                f"{TEACHER_MAX_RETRIES}: {exc}"
            )

            time.sleep(wait)

    raise RuntimeError(
        f"Teacher failed after retries: {last_error}"
    )

## 16. Validate teacher payload against the closed contract

In [ ]:
TEACHER_REQUIRED_KEYS = {
    "task_type",
    "subtype",
    "target_series",
    "target_category",
    "relevant_values",
    "operation",
    "calculation",
    "final_answer",
    "confidence",
}


def validate_teacher_payload(payload):
    errors = []

    if not isinstance(payload, dict):
        return False, ["PAYLOAD_NOT_DICT"]

    missing = TEACHER_REQUIRED_KEYS - set(payload)

    if missing:
        errors.append(
            "MISSING_KEYS:" + ",".join(sorted(missing))
        )

    if payload.get("task_type") not in TASK_TYPES:
        errors.append("INVALID_TASK_TYPE")

    if payload.get("subtype") not in SUBTYPES:
        errors.append("INVALID_SUBTYPE")

    if payload.get("operation") not in OPERATIONS:
        errors.append("INVALID_OPERATION")

    if not isinstance(payload.get("relevant_values"), list):
        errors.append("RELEVANT_VALUES_NOT_LIST")

    try:
        confidence = float(payload.get("confidence"))
        if not 0 <= confidence <= 1:
            errors.append("INVALID_CONFIDENCE_RANGE")
    except Exception:
        errors.append("INVALID_CONFIDENCE")

    if not normalize_text(payload.get("final_answer", "")):
        errors.append("EMPTY_FINAL_ANSWER")

    return len(errors) == 0, errors

## 17. Validate annotation content

A sample can be:

- `VALIDATED`
- `REVIEW_REPRESENTATION`
- `REVIEW_CONFLICT`
- `LOW_CONFIDENCE`
- `INVALID_SCHEMA`
- `TEACHER_ERROR`

Representation-equivalent answers are kept separate from semantic conflicts.

In [ ]:
def validate_annotation(base_record, teacher):
    schema_valid, schema_errors = validate_teacher_payload(teacher)

    if not schema_valid:
        return {
            "status": "INVALID_SCHEMA",
            "answer_relation": None,
            "arithmetic_valid": None,
            "validation_flags": schema_errors,
        }

    confidence = float(teacher["confidence"])

    if confidence < TEACHER_MIN_CONFIDENCE:
        return {
            "status": "LOW_CONFIDENCE",
            "answer_relation": None,
            "arithmetic_valid": None,
            "validation_flags": ["LOW_TEACHER_CONFIDENCE"],
        }

    relation = classify_answer_relation(
        base_record["dataset_answer"],
        teacher["final_answer"],
    )

    arithmetic_valid = None
    arithmetic_flag = []

    values = teacher.get("relevant_values", [])
    operation = teacher.get("operation", "none")

    if operation not in {"none", "lookup", "comparison", "min_max", "multi_step"}:
        recomputed = recompute_operation(values, operation)
        teacher_numeric = parse_number(teacher["final_answer"])

        if recomputed is not None and teacher_numeric is not None:
            arithmetic_valid = (
                abs(recomputed - teacher_numeric["value"]) <= 1e-5
            )

            if not arithmetic_valid:
                arithmetic_flag.append("ARITHMETIC_RECOMPUTE_FAILED")

    if arithmetic_flag:
        return {
            "status": "REVIEW_CONFLICT",
            "answer_relation": relation,
            "arithmetic_valid": arithmetic_valid,
            "validation_flags": arithmetic_flag,
        }

    if relation in {"EXACT_MATCH", "NUMERIC_MATCH"}:
        status = "VALIDATED"

    elif relation in {
        "PERCENT_PROPORTION_EQUIVALENT",
        "POSSIBLE_SCALE_EQUIVALENT",
    }:
        status = "REVIEW_REPRESENTATION"

    else:
        status = "REVIEW_CONFLICT"

    return {
        "status": status,
        "answer_relation": relation,
        "arithmetic_valid": arithmetic_valid,
        "validation_flags": [],
    }

## 18. Checkpoint helpers

In [ ]:
def json_safe(value):
    if isinstance(value, (str, int, float, bool, type(None))):
        return value

    if isinstance(value, list):
        return [json_safe(x) for x in value]

    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}

    return str(value)


def save_jsonl(records, path):
    with open(path, "w", encoding="utf-8") as f:
        for record in records:
            f.write(
                json.dumps(
                    json_safe(record),
                    ensure_ascii=False,
                )
                + "\n"
            )


def load_jsonl(path):
    path = Path(path)

    if not path.exists():
        return []

    records = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))

    return records

# 19. Generate teacher annotations

Recommended progression:

```text
50 samples
↓
inspect
↓
500 samples
↓
inspect
↓
full train
```

In [ ]:
assert_teacher_available()
print("Teacher endpoint preflight passed.")

existing_records = load_jsonl(ANNOTATION_CHECKPOINT)

completed_ids = {
    record["sample_id"]
    for record in existing_records
}

annotation_records = list(existing_records)

build_n = len(train_ds)

if PHASE2A_BUILD_LIMIT is not None:
    build_n = min(PHASE2A_BUILD_LIMIT, len(train_ds))

print("Resuming from:", len(annotation_records))
print("Target build size:", build_n)

for i in range(build_n):
    base_record = build_base_record(train_ds[i], i)
    sample_id = base_record["sample_id"]

    if sample_id in completed_ids:
        continue

    if (
        not base_record["image_present"]
        or not base_record["question"]
        or not base_record["dataset_answer"]
    ):
        base_record.update({
            "annotation_status": "DROP",
            "validation_flags": ["MISSING_REQUIRED_FIELD"],
        })

        annotation_records.append(base_record)
        completed_ids.add(sample_id)
        continue

    try:
        teacher = call_teacher(
            base_record,
            train_ds[i]["image"],
        )

        validation = validate_annotation(
            base_record,
            teacher,
        )

        base_record.update({
            "teacher_task_type": teacher.get("task_type"),
            "teacher_subtype": teacher.get("subtype"),
            "target_series": teacher.get("target_series"),
            "target_category": teacher.get("target_category"),
            "relevant_values": teacher.get("relevant_values"),
            "operation": teacher.get("operation"),
            "calculation": teacher.get("calculation"),
            "teacher_final_answer": teacher.get("final_answer"),
            "teacher_confidence": teacher.get("confidence"),
            "answer_relation": validation["answer_relation"],
            "arithmetic_valid": validation["arithmetic_valid"],
            "annotation_status": validation["status"],
            "validation_flags": validation["validation_flags"],
        })

    except Exception as exc:
        base_record.update({
            "annotation_status": "TEACHER_ERROR",
            "validation_flags": [
                f"TEACHER_ERROR:{type(exc).__name__}"
            ],
        })

    annotation_records.append(base_record)
    completed_ids.add(sample_id)

    if len(annotation_records) % SAVE_EVERY == 0:
        save_jsonl(
            annotation_records,
            ANNOTATION_CHECKPOINT,
        )
        print(
            f"Saved {len(annotation_records)}/{build_n}"
        )

save_jsonl(annotation_records, ANNOTATION_CHECKPOINT)

print("Annotation stage finished.")

## 20. Inspect annotation results

In [ ]:
annotations_df = pd.DataFrame(annotation_records)

print(
    annotations_df["annotation_status"]
    .value_counts(dropna=False)
)

display(
    annotations_df[
        [
            "dataset_index",
            "question",
            "dataset_answer",
            "teacher_task_type",
            "teacher_subtype",
            "relevant_values",
            "operation",
            "calculation",
            "teacher_final_answer",
            "answer_relation",
            "teacher_confidence",
            "annotation_status",
            "validation_flags",
        ]
    ].head(30)
)

## 21. Annotation quality report

In [ ]:
def annotation_quality_report(df):
    total = len(df)

    status_counts = (
        df["annotation_status"]
        .value_counts(dropna=False)
        .to_dict()
    )

    validated = int(
        (df["annotation_status"] == "VALIDATED").sum()
    )

    representation_review = int(
        (df["annotation_status"] == "REVIEW_REPRESENTATION").sum()
    )

    conflicts = int(
        (df["annotation_status"] == "REVIEW_CONFLICT").sum()
    )

    teacher_errors = int(
        (df["annotation_status"] == "TEACHER_ERROR").sum()
    )

    return {
        "total_records": int(total),
        "status_counts": status_counts,
        "validated_count": validated,
        "validated_rate": validated / total if total else 0.0,
        "representation_review_count": representation_review,
        "representation_review_rate":
            representation_review / total if total else 0.0,
        "conflict_count": conflicts,
        "conflict_rate": conflicts / total if total else 0.0,
        "teacher_error_count": teacher_errors,
        "teacher_error_rate": teacher_errors / total if total else 0.0,
        "task_distribution":
            df["teacher_task_type"]
            .value_counts(dropna=False)
            .to_dict(),
        "subtype_distribution":
            df["teacher_subtype"]
            .value_counts(dropna=False)
            .to_dict(),
    }


annotation_report = annotation_quality_report(annotations_df)

print(
    json.dumps(
        annotation_report,
        indent=2,
    )
)

## 22. Development quality gates

These are engineering gates for deciding whether the annotation pipeline is ready to scale.

They are not benchmark claims.

In [ ]:
MIN_VALIDATED_RATE = 0.70
MAX_CONFLICT_RATE = 0.20
MAX_TEACHER_ERROR_RATE = 0.05

validated_rate = annotation_report["validated_rate"]
conflict_rate = annotation_report["conflict_rate"]
teacher_error_rate = annotation_report["teacher_error_rate"]

ANNOTATION_AUTO_GATE = (
    validated_rate >= MIN_VALIDATED_RATE
    and conflict_rate <= MAX_CONFLICT_RATE
    and teacher_error_rate <= MAX_TEACHER_ERROR_RATE
)

print("Validated rate:", f"{validated_rate:.1%}")
print("Conflict rate:", f"{conflict_rate:.1%}")
print("Teacher error rate:", f"{teacher_error_rate:.1%}")
print("Annotation auto gate:", ANNOTATION_AUTO_GATE)

# 23. Manual Audit

The audit sample should include:

- validated examples from every task
- representation mismatches
- semantic conflicts
- teacher errors / invalid schema

In [ ]:
AUDIT_PER_TASK = 15
AUDIT_REVIEW_CASES = 40

audit_parts = []

validated_df = annotations_df[
    annotations_df["annotation_status"] == "VALIDATED"
].copy()

for task in sorted(
    validated_df["teacher_task_type"]
    .dropna()
    .unique()
):
    subset = validated_df[
        validated_df["teacher_task_type"] == task
    ]

    n = min(AUDIT_PER_TASK, len(subset))

    if n:
        audit_parts.append(
            subset.sample(
                n=n,
                random_state=SEED,
            )
        )

review_df = annotations_df[
    annotations_df["annotation_status"].isin(
        [
            "REVIEW_REPRESENTATION",
            "REVIEW_CONFLICT",
            "LOW_CONFIDENCE",
            "INVALID_SCHEMA",
            "TEACHER_ERROR",
        ]
    )
]

if len(review_df):
    audit_parts.append(
        review_df.sample(
            n=min(AUDIT_REVIEW_CASES, len(review_df)),
            random_state=SEED,
        )
    )

if audit_parts:
    audit_df = (
        pd.concat(audit_parts, ignore_index=True)
        .drop_duplicates(subset=["sample_id"])
    )
else:
    audit_df = annotations_df.head(0)

AUDIT_CSV = (
    AUDIT_DIR
    / "phase2a_teacher_annotation_audit.csv"
)

audit_columns = [
    "sample_id",
    "dataset_index",
    "question",
    "dataset_answer",
    "teacher_task_type",
    "teacher_subtype",
    "target_series",
    "target_category",
    "relevant_values",
    "operation",
    "calculation",
    "teacher_final_answer",
    "answer_relation",
    "teacher_confidence",
    "annotation_status",
    "validation_flags",
]

audit_df[audit_columns].to_csv(
    AUDIT_CSV,
    index=False,
)

print("Audit samples:", len(audit_df))
print("Saved:", AUDIT_CSV)

display(audit_df[audit_columns].head(100))

## 24. Manual audit approval

Set this to `True` only after actually reviewing the exported audit CSV.

In [ ]:
MANUAL_AUDIT_APPROVED = False

print("Manual audit approved:", MANUAL_AUDIT_APPROVED)

# 25. Build SFT-Eligible Pool

Initial training policy:

### Automatically eligible
- `VALIDATED`

### Not automatically eligible
- `REVIEW_REPRESENTATION`
- `REVIEW_CONFLICT`
- `LOW_CONFIDENCE`
- `INVALID_SCHEMA`
- `TEACHER_ERROR`

Representation-review samples can be incorporated later after explicit normalization rules are approved.

In [ ]:
eligible_df = annotations_df[
    annotations_df["annotation_status"] == "VALIDATED"
].copy()

eligible_df = eligible_df[
    eligible_df["teacher_task_type"].isin(TASK_TYPES)
]

print("Eligible samples:", len(eligible_df))

display(
    eligible_df["teacher_task_type"]
    .value_counts()
)

display(
    eligible_df["teacher_task_type"]
    .value_counts(normalize=True)
)

## 26. Curriculum selection

Do not destroy large amounts of clean data merely to force an exact 45/30/15/10 split.

This notebook uses a **soft curriculum**:

- keep all validated samples
- compute sampling weights relative to the target curriculum
- Phase 2B may use these weights or weighted sampling

This avoids the previous 245 → 50 sample collapse caused by hard undersampling.

In [ ]:
observed_share = (
    eligible_df["teacher_task_type"]
    .value_counts(normalize=True)
    .to_dict()
)

sampling_weights = {}

for task, target_share in CURRICULUM_TARGETS.items():
    observed = observed_share.get(task, 0.0)

    if observed > 0:
        sampling_weights[task] = target_share / observed
    else:
        sampling_weights[task] = 0.0

if len(eligible_df):
    eligible_df["curriculum_weight"] = (
        eligible_df["teacher_task_type"]
        .map(sampling_weights)
        .fillna(0.0)
    )
else:
    eligible_df["curriculum_weight"] = pd.Series(dtype=float)

print("Sampling weights:")
print(json.dumps(sampling_weights, indent=2))

## 27. Build final SFT target text

In [ ]:
def format_sft_target(row):
    lines = []

    if row.get("target_series"):
        lines.append(
            f"Target series: {row['target_series']}"
        )

    if row.get("target_category"):
        lines.append(
            f"Target category: {row['target_category']}"
        )

    values = row.get("relevant_values")

    if isinstance(values, list) and values:
        lines.append(
            "Relevant values: "
            + ", ".join(map(str, values))
        )

    operation = row.get("operation")

    if operation and operation != "none":
        lines.append(
            f"Operation: {operation}"
        )

    if row.get("calculation"):
        lines.append(
            f"Calculation: {row['calculation']}"
        )

    # Keep the dataset answer as the final SFT answer for validated cases.
    lines.append(
        f"Answer: {row['dataset_answer']}"
    )

    return "\n".join(lines)


if len(eligible_df):
    eligible_df["sft_target"] = eligible_df.apply(
        format_sft_target,
        axis=1,
    )
else:
    eligible_df["sft_target"] = pd.Series(dtype=str)

display(
    eligible_df[
        [
            "question",
            "teacher_task_type",
            "teacher_subtype",
            "sft_target",
        ]
    ].head(20)
)

## 28. Final SFT row validation

In [ ]:
def final_sft_row_valid(row):
    if row["teacher_task_type"] not in TASK_TYPES:
        return False

    if not normalize_text(row["question"]):
        return False

    if not normalize_text(row["dataset_answer"]):
        return False

    if not normalize_text(row["sft_target"]):
        return False

    if row["annotation_status"] != "VALIDATED":
        return False

    return True


if len(eligible_df):
    eligible_df["final_validation"] = eligible_df.apply(
        final_sft_row_valid,
        axis=1,
    )

    print(
        eligible_df["final_validation"]
        .value_counts()
    )

    assert eligible_df["final_validation"].all()
else:
    eligible_df["final_validation"] = pd.Series(dtype=bool)
    print("No eligible SFT rows yet.")

## 29. Export SFT-ready artifacts

In [ ]:
SFT_READY_JSONL = (
    RESULTS_DIR
    / f"phase2a_pilot_{PHASE2A_BUILD_LIMIT}_sft_candidate.jsonl"
)

SFT_READY_CSV = (
    RESULTS_DIR
    / f"phase2a_pilot_{PHASE2A_BUILD_LIMIT}_sft_candidate_metadata.csv"
)

FINAL_REPORT_JSON = (
    RESULTS_DIR
    / f"phase2a_pilot_{PHASE2A_BUILD_LIMIT}_report.json"
)

export_columns = [
    "sample_id",
    "dataset_index",
    "question",
    "dataset_answer",
    "teacher_task_type",
    "teacher_subtype",
    "target_series",
    "target_category",
    "relevant_values",
    "operation",
    "calculation",
    "teacher_confidence",
    "curriculum_weight",
    "sft_target",
]

if len(eligible_df):
    with open(SFT_READY_JSONL, "w", encoding="utf-8") as f:
        for _, row in eligible_df.iterrows():
            record = {
                key: json_safe(row.get(key))
                for key in export_columns
            }

            f.write(
                json.dumps(
                    record,
                    ensure_ascii=False,
                )
                + "\n"
            )

    eligible_df[export_columns].to_csv(
        SFT_READY_CSV,
        index=False,
    )

final_report = {
    "raw_chartqa_train_samples": int(len(train_ds)),
    "processed_samples": int(len(annotations_df)),
    "validated_samples": int(len(eligible_df)),
    "annotation_status":
        annotations_df["annotation_status"]
        .value_counts(dropna=False)
        .to_dict(),
    "task_distribution":
        eligible_df["teacher_task_type"]
        .value_counts()
        .to_dict(),
    "sampling_weights": sampling_weights,
    "teacher_min_confidence": TEACHER_MIN_CONFIDENCE,
    "manual_audit_approved": bool(MANUAL_AUDIT_APPROVED),
    "phase1_validation_used_for_training": False,
}

FINAL_REPORT_JSON.write_text(
    json.dumps(
        final_report,
        indent=2,
    ),
    encoding="utf-8",
)

print("SFT JSONL:", SFT_READY_JSONL)
print("Metadata CSV:", SFT_READY_CSV)
print("Final report:", FINAL_REPORT_JSON)

# 30. Pilot and full-production gates

A 50- or 500-sample run is a pilot, not full Phase 2A completion.

The 500-sample pilot should report `READY_FOR_MANUAL_AUDIT` after its automated artifacts are complete. Full completion remains a separate, honest gate:

```python
PHASE2A_BUILD_LIMIT = None
MANUAL_AUDIT_APPROVED = True
```

only after the full annotation run and actual audit are complete.

In [ ]:
PILOT_BUILD_COMPLETE = (
    PHASE2A_BUILD_LIMIT == PILOT_TARGET_SAMPLES
    and len(annotations_df) == min(PILOT_TARGET_SAMPLES, len(train_ds))
)

PILOT_ARTIFACTS_READY = (
    PILOT_BUILD_COMPLETE
    and ANNOTATION_AUTO_GATE
    and AUDIT_CSV.exists()
    and SFT_READY_JSONL.exists()
    and SFT_READY_CSV.exists()
    and FINAL_REPORT_JSON.exists()
    and len(eligible_df) > 0
)

if not PILOT_ARTIFACTS_READY:
    PILOT_500_STATUS = "INCOMPLETE"
elif MANUAL_AUDIT_APPROVED:
    PILOT_500_STATUS = "APPROVED_FOR_SFT_EXPERIMENT"
else:
    PILOT_500_STATUS = "READY_FOR_MANUAL_AUDIT"

FULL_BUILD_REQUESTED = PHASE2A_BUILD_LIMIT is None
FULL_ANNOTATION_COMPLETE = (
    FULL_BUILD_REQUESTED and len(annotations_df) == len(train_ds)
)
FULL_AUDIT_COMPLETE = (
    FULL_ANNOTATION_COMPLETE
    and ANNOTATION_AUTO_GATE
    and AUDIT_CSV.exists()
    and MANUAL_AUDIT_APPROVED
)
SFT_EXPORT_COMPLETE = (
    FULL_AUDIT_COMPLETE
    and SFT_READY_JSONL.exists()
    and SFT_READY_CSV.exists()
    and FINAL_REPORT_JSON.exists()
    and len(eligible_df) > 0
)

FULL_PHASE2A_STATUS = (
    "NOT_REQUESTED" if not FULL_BUILD_REQUESTED
    else "COMPLETE" if SFT_EXPORT_COMPLETE
    else "INCOMPLETE"
)

PHASE2A_STATUS = {
    "implementation": "READY",
    "pilot_500": PILOT_500_STATUS,
    "pilot_processed_samples": int(len(annotations_df)),
    "pilot_validated_samples": int(len(eligible_df)),
    "artifacts_root": str(RESULTS_DIR),
    "full_phase2a": FULL_PHASE2A_STATUS,
}

print(json.dumps(PHASE2A_STATUS, indent=2))
print("\nNext action:", (
    "review the audit CSV and set MANUAL_AUDIT_APPROVED = True"
    if PILOT_500_STATUS == "READY_FOR_MANUAL_AUDIT"
    else "resolve the incomplete pilot gate"
))

# 31. Recommended execution order

```text
1. Run Sections 1–14
2. Configure teacher
3. PHASE2A_BUILD_LIMIT = 50
4. Run annotation + validation
5. Inspect status distribution and 20–30 examples
6. Increase to 500
7. Inspect validated/conflict/representation rates
8. Refine taxonomy or validator if necessary
9. Set PHASE2A_BUILD_LIMIT = None
10. Run full annotation
11. Review phase2a_teacher_annotation_audit.csv
12. Set MANUAL_AUDIT_APPROVED = True only if audit passes
13. Export final SFT-ready dataset
14. Confirm PHASE 2A COMPLETE: True
15. Start Phase 2B QLoRA
```

### Why this version is different

The previous pipeline tried to classify ChartQA questions with regex first.

This version makes the teacher the **primary structured annotator**, while deterministic code becomes the validator.

That gives a cleaner architecture:

```text
image + question
      ↓
teacher annotation
      ↓
closed schema
      ↓
answer relation check
      ↓
arithmetic check
      ↓
confidence / conflict filtering
      ↓
audited SFT dataset
```